# React 框架 · 第 02 天

## 表单控制、组件通信与 Hooks

本篇依据已下载的 React 基础 day02 MD 笔记、PPT 和随堂代码整理。重点学习受控表单、组件之间的数据传递、`useEffect` 副作用管理、自定义 Hook 以及 Hooks 使用规则，并继续优化 B 站评论案例。

## 一、学习目标

学完本篇后，应能：

- 区分受控表单和非受控表单，并选择合适的实现方式。
- 使用 `value`、`onChange` 和 `useRef` 完成表单数据读取。
- 掌握父传子、子传父、兄弟组件和跨层组件通信。
- 使用 `children` 接收组件标签之间的嵌套内容。
- 使用 `useEffect` 处理请求、定时器等副作用，并正确清理。
- 把可复用逻辑封装为自定义 Hook。
- 遵守 Hooks 只能在组件顶层调用的规则。
- 用接口请求、自定义 Hook 和子组件拆分优化评论案例。

## 二、表单控制：受控绑定

受控组件由 React 状态控制表单值，输入框的 `value` 来自状态，用户输入通过 `onChange` 回写状态：

```jsx
import { useState } from 'react'

function App() {
  const [value, setValue] = useState('')

  return (
    <input
      type="text"
      value={value}
      onChange={event => setValue(event.target.value)}
    />
  )
}
```

数据流是单向的：状态决定输入框显示什么，输入事件触发 setter 更新状态。这样做便于校验、格式化、提交和重置。

常见表单写法：

```jsx
const [form, setForm] = useState({ username: '', password: '' })

<input
  value={form.username}
  onChange={e => setForm({ ...form, username: e.target.value })}
/>
```

修改对象状态时要创建新对象，避免直接修改 `form.username`。

## 三、非受控绑定：通过 ref 读取 DOM

非受控组件把真实值保存在 DOM 中，需要时通过 `useRef` 取得 DOM 节点：

```jsx
import { useRef } from 'react'

function App() {
  const inputRef = useRef(null)

  const readValue = () => {
    console.log(inputRef.current.value)
  }

  return (
    <>
      <input ref={inputRef} type="text" />
      <button onClick={readValue}>读取</button>
    </>
  )
}
```

### 如何选择

- 需要实时校验、联动、格式化、受状态统一管理时，优先使用受控绑定。
- 只在提交时读取一次，或接入原生表单、第三方 DOM 插件时，可以考虑非受控绑定。

不要在同一个输入框上同时使用 `value` 和 `defaultValue`，否则容易出现受控与非受控切换警告。

## 四、B 站评论案例：发布评论

在已有评论列表基础上，新增评论输入框和发布功能：

```jsx
import { useState } from 'react'
import dayjs from 'dayjs'
import { v4 as uuid } from 'uuid'

function CommentForm({ onAdd }) {
  const [content, setContent] = useState('')

  const handleSubmit = () => {
    const value = content.trim()
    if (!value) return

    onAdd({
      rpid: uuid(),
      content: value,
      ctime: dayjs().format('MM-DD HH:mm'),
      like: 0,
      user: { uid: 'current-user', uname: '当前用户', avatar: '' },
    })
    setContent('')
  }

  return (
    <div>
      <textarea
        value={content}
        onChange={e => setContent(e.target.value)}
        placeholder="发一条友善的评论"
      />
      <button onClick={handleSubmit}>发布</button>
    </div>
  )
}
```

`uuid` 用于生成稳定、唯一的评论 id，`dayjs` 用于格式化发布时间。发布成功后由父组件把新评论放入列表，再由列表状态驱动页面更新。

## 五、组件通信总览

组件通信就是组件之间的数据传递。通信方式取决于组件之间的嵌套关系：

| 关系 | 常用方式 | 核心思路 |
| --- | --- | --- |
| 父 → 子 | props | 父组件在子组件标签上绑定属性 |
| 子 → 父 | 回调函数 props | 父组件把函数传给子组件，子组件调用并传参 |
| 兄弟组件 | 状态提升 | 共同父组件保存状态，再分别传给两个子组件 |
| 跨层组件 | Context | 顶层 Provider 提供数据，底层 useContext 消费 |

组件通信的原则是：数据由拥有它的组件管理，通过 props 或上下文向需要的地方传递，子组件不要直接修改父组件的数据。

## 六、父传子：props

父组件在子组件标签上绑定属性，子组件通过 `props` 参数接收：

```jsx
function Son(props) {
  return <div>{props.name}</div>
}

function App() {
  const name = 'this is app name'
  return <Son name={name} />
}
```

也可以直接解构：

```jsx
function Son({ name, age, user }) {
  return <p>{name} - {age} - {user.nickname}</p>
}
```

### props 的特点

- 可以传递字符串、数字、布尔值、数组、对象、函数和 JSX 等合法数据。
- `props` 是只读对象，子组件不能直接修改它。
- 父组件的数据只能由父组件修改，子组件需要通过回调通知父组件。

```jsx
<Son
  title="标题"
  count={3}
  user={{ name: 'jack' }}
  onSave={() => console.log('save')}
/>
```

## 七、特殊的 `children` prop

当内容嵌套在组件标签内部时，React 会把它作为 `children` 传给组件：

```jsx
function Panel({ title, children }) {
  return (
    <section className="panel">
      <h3>{title}</h3>
      <div>{children}</div>
    </section>
  )
}

function App() {
  return (
    <Panel title="用户信息">
      <p>这里是由父组件传入的内容</p>
      <button>操作</button>
    </Panel>
  )
}
```

`children` 适合实现布局容器、弹窗、卡片、面板等“外壳组件”。组件负责结构和样式，具体内容由使用者传入。

## 八、子传父：通过回调函数

React 没有直接的子传父语法，通常由父组件定义函数并通过 props 传给子组件，子组件调用函数传递数据：

```jsx
function Son({ onGetMsg }) {
  const sonMsg = 'this is son msg'

  return (
    <button onClick={() => onGetMsg(sonMsg)}>
      发送给父组件
    </button>
  )
}

function App() {
  const getMsg = msg => {
    console.log('父组件收到：', msg)
  }

  return <Son onGetMsg={getMsg} />
}
```

子组件只负责在合适的时机调用回调，父组件决定收到数据后如何处理。表单提交、删除确认、子组件选择结果等场景都常用这种方式。

## 九、兄弟组件通信：状态提升

兄弟组件不能直接通信，需要把共享状态提升到共同的父组件：

```jsx
import { useState } from 'react'

function A({ onGetAName }) {
  const name = 'this is A name'
  return <button onClick={() => onGetAName(name)}>A 发送</button>
}

function B({ name }) {
  return <div>B 收到：{name}</div>
}

function App() {
  const [name, setName] = useState('')

  return (
    <>
      <A onGetAName={setName} />
      <B name={name} />
    </>
  )
}
```

流程是：A 通过回调把数据交给 App，App 更新自己的状态，再把最新状态通过 props 传给 B。这样数据来源清晰，兄弟组件之间不会产生隐式依赖。

## 十、跨层通信：Context

当数据需要跨过多层组件传递时，逐层传 props 会产生“prop drilling”。可以使用 Context：

```jsx
import { createContext, useContext } from 'react'

const MsgContext = createContext(null)

function A() {
  return <B />
}

function B() {
  const msg = useContext(MsgContext)
  return <div>底层组件收到：{msg}</div>
}

function App() {
  return (
    <MsgContext.Provider value="this is app msg">
      <A />
    </MsgContext.Provider>
  )
}
```

Provider 负责提供 `value`，任意后代组件通过 `useContext(MsgContext)` 消费数据。Context 适合主题、当前用户、语言、全局配置等跨层数据；局部、频繁变化的数据仍然优先使用 props 和状态提升。

## 十一、副作用与 `useEffect`

副作用是指不是单纯由组件计算 UI 产生，而是需要和组件外部世界交互的操作，例如：

- 请求接口、订阅 WebSocket。
- 开启定时器、监听窗口事件。
- 操作非 React 管理的 DOM。
- 把数据同步到浏览器存储。

`useEffect` 在组件渲染后执行副作用函数：

```jsx
import { useEffect, useState } from 'react'

function ChannelList() {
  const [channels, setChannels] = useState([])

  useEffect(() => {
    fetch('http://geek.itheima.net/v1_0/channels')
      .then(response => response.json())
      .then(result => setChannels(result.data.channels))
  }, [])

  return (
    <ul>
      {channels.map(channel => (
        <li key={channel.id}>{channel.name}</li>
      ))}
    </ul>
  )
}
```

接口请求属于“由渲染触发”的操作，不应直接写在组件函数体中，否则每次重新渲染都可能重复请求。

## 十二、`useEffect` 依赖项

依赖项数组决定副作用函数的执行时机：

| 写法 | 执行时机 |
| --- | --- |
| `useEffect(fn)` | 初次渲染和每次更新后执行 |
| `useEffect(fn, [])` | 只在初次渲染后执行一次 |
| `useEffect(fn, [value])` | 初次渲染执行，`value` 改变后再次执行 |

```jsx
useEffect(() => {
  document.title = `数量：${count}`
}, [count])
```

依赖项要包含副作用函数中使用、且来自组件作用域的变化值。请求依赖用户 id 时，应把 `userId` 放进依赖数组；否则切换用户后可能仍然显示旧数据。开发环境下某些 React 配置可能让副作用在开发期被额外检查，不能把这种检查当成生产环境的重复执行逻辑。

## 十三、清除副作用

如果副作用开启了定时器、事件监听或订阅，就需要在组件卸载或依赖变化前清理：

```jsx
import { useEffect, useState } from 'react'

function Son() {
  useEffect(() => {
    const timer = setInterval(() => {
      console.log('定时器执行中...')
    }, 1000)

    return () => {
      clearInterval(timer)
    }
  }, [])

  return <div>定时器组件</div>
}

function App() {
  const [show, setShow] = useState(true)

  return (
    <>
      {show && <Son />}
      <button onClick={() => setShow(false)}>卸载 Son</button>
    </>
  )
}
```

`useEffect` 返回的函数就是清理函数。React 会在组件卸载时执行它；当依赖发生变化时，也会先清理上一次副作用，再执行新的副作用。清理可以避免定时器叠加、事件重复绑定和请求结果污染。

## 十四、自定义 Hook

自定义 Hook 是以 `use` 开头的函数，用于封装和复用状态逻辑。以布尔切换为例：

```jsx
import { useState } from 'react'

function useToggle(initialValue = true) {
  const [value, setValue] = useState(initialValue)
  const toggle = () => setValue(prevValue => !prevValue)

  return { value, toggle }
}

function App() {
  const { value, toggle } = useToggle()

  return (
    <>
      {value && <div>this is div</div>}
      <button onClick={toggle}>切换</button>
    </>
  )
}
```

封装步骤：

1. 声明一个以 `use` 开头的函数。
2. 在函数内部编写可复用的状态和副作用逻辑。
3. 返回调用方需要的状态、数据和回调函数。
4. 在组件中执行自定义 Hook，并解构使用返回值。

自定义 Hook 共享的是逻辑，不是同一份状态；每个组件调用一次，就会拥有自己独立的状态实例。

## 十五、Hooks 使用规则

1. Hooks 只能在 React 函数组件或其他自定义 Hook 中调用。
2. Hooks 只能在组件函数的顶层调用，不能放进 `if`、`for`、`while` 或嵌套函数中。

```jsx
function App({ enabled }) {
  // 正确：每次渲染都按照相同顺序调用
  const [count, setCount] = useState(0)

  if (!enabled) {
    return null
  }

  return <button onClick={() => setCount(count + 1)}>{count}</button>
}
```

下面这种写法是错误的：

```jsx
// 错误：Hook 不能放在条件分支中
if (enabled) {
  const [count, setCount] = useState(0)
}
```

React 依赖 Hooks 调用顺序保存状态。条件调用会改变顺序，导致状态对应关系混乱。可以让 Hook 始终调用，再在 Hook 内部或返回的 JSX 中处理条件。

## 十六、优化 B 站评论案例的结构

第二天的案例升级方向：

- 评论列表从接口请求，不再完全依赖静态数组。
- 把请求、加载和错误处理封装到自定义 Hook。
- 把每条评论抽成独立组件，降低 `App` 复杂度。

```jsx
function useComments(articleId) {
  const [list, setList] = useState([])
  const [loading, setLoading] = useState(false)
  const [error, setError] = useState(null)

  useEffect(() => {
    let ignore = false
    setLoading(true)

    fetch(`/api/articles/${articleId}/comments`)
      .then(response => {
        if (!response.ok) throw new Error('请求失败')
        return response.json()
      })
      .then(data => {
        if (!ignore) setList(data)
      })
      .catch(err => {
        if (!ignore) setError(err)
      })
      .finally(() => {
        if (!ignore) setLoading(false)
      })

    return () => {
      ignore = true
    }
  }, [articleId])

  return { list, setList, loading, error }
}
```

拆分后的 `CommentItem` 只接收 `item` 和 `onDelete`，`CommentList` 负责遍历，`CommentForm` 负责新增，页面组件负责组合。这样每个组件的数据来源、职责和更新入口都更清楚。

## 十七、常见错误与排查顺序

1. **输入框无法输入**：使用了 `value` 却没有 `onChange`，受控输入需要同时提供状态和更新函数。
2. **受控/非受控警告**：初始值保持一致，文本输入可用 `''`，不要从 `undefined` 变成字符串。
3. **子组件改不了 props**：props 是只读的，改动应通过回调通知父组件。
4. **接口请求重复**：确认请求放在 `useEffect` 中，并检查依赖项是否正确。
5. **定时器越来越多**：在 effect 返回清理函数，卸载或依赖变化时清除。
6. **Hook 报调用顺序错误**：把 Hook 移到组件顶层，不要放在条件、循环和回调里。
7. **评论更新不生效**：不要直接修改原数组，使用 `filter`、展开运算符或函数式 setter 生成新数据。
8. **旧请求覆盖新请求**：依赖变化时清理旧请求或使用取消/忽略标记，避免竞态。

排查时按照“输入值 → 事件 → setter → 状态 → effect → 渲染”的链路逐步确认。

## 十八、本日总结

- 受控组件让 React 状态掌管表单值，非受控组件通过 ref 读取 DOM。
- 父传子用 props，子传父用回调，兄弟组件用状态提升，跨层组件可用 Context。
- `children` 让容器组件接收嵌套内容。
- `useEffect` 用于处理由渲染触发的副作用，依赖数组决定执行时机，返回函数负责清理。
- 自定义 Hook 用于复用状态逻辑，但每个组件调用后仍拥有独立状态。
- Hooks 必须在组件或自定义 Hook 的顶层调用。
- B 站评论案例把表单、请求、组件通信、列表状态和自定义 Hook 串联起来。

下一步可以继续学习路由、异步状态、项目级数据管理和更完整的 React 工程实践。